# Import Packages

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
from glob import glob

# work directory

In [2]:
wrk_directory ="C:/Users/l_v_v/Documents/GitHub/time_series_curuai/datasets/Parameters Time series/TSS Modeling"

# import Data

In [65]:
paths = sorted(glob(os.path.join(wrk_directory,"*metrics*.csv")))
cv_path = []
metric_path = []
for path in paths:
    if "cv" in path:
        cv_path.append(path)
    else:
        metric_path.append(path)

In [66]:
cv_dfs= []
m_dfs = []
period_cv = []
period_metric = []

for metric, cv_metric in zip(metric_path, cv_path):
    df_metric = pd.read_csv(metric)
    df_metric['rmse'] = np.sqrt(df_metric['mse'])
    df_cv = pd.read_csv(cv_metric)
    df_cv['rmse'] = np.sqrt(df_cv['mse'])
    if 'Unnamed: 0' in df_metric.columns:
        df_metric = df_metric.drop(columns=['Unnamed: 0'])
    if 'Unnamed: 0' in df_cv.columns:
        df_cv = df_cv.drop(columns=['Unnamed: 0'])
    
    cv_dfs.append(df_cv)
    m_dfs.append(df_metric)
final_cv_df = pd.concat(cv_dfs, ignore_index=True)
final_metric_df = pd.concat(m_dfs, ignore_index=True)  

In [67]:
period_cv = final_cv_df.dropna(subset='water_period')
period_m = final_metric_df.dropna(subset='water_period')
period_cv

,Model,Group,Feature,Params,r2,mae,mse,mape,exp_var,rmse,water_period
69,ols,single_band,nir,NaN,0.750459,13.136463,4.484970e+02,0.286021,0.403573,21.177748,R
70,ols,single_band,red,NaN,0.468110,11.932755,4.286936e+02,0.256130,0.315162,20.704917,R
71,ols,single_band,nir_red_ratio,NaN,0.908369,13.853791,5.581768e+02,0.347170,0.400018,23.625766,R
72,ols,single_band,red_nir_ratio,NaN,0.542325,13.245558,4.979487e+02,0.309173,0.088449,22.314763,R
73,ols,multi_band,nir_red,NaN,0.688434,12.612278,4.277580e+02,0.267919,0.464074,20.682311,R
...,...,...,...,...,...,...,...,...,...,...,...
304,polynomial3,multi_band,nir_red_green,NaN,0.317686,66.027041,7.853916e+03,0.536376,0.343744,88.622324,LW
305,polynomial3,multi_band,all_bands,NaN,99.808216,285.634545,8.580621e+05,4.007965,88.781248,926.316436,LW
306,polynomial3,multi_band,location,NaN,548.006221,885.275299,3.527450e+06,9.384929,493.743708,1878.150577,LW
307,polynomial3,multi_band,period,NaN,99.808216,285.634545,8.580621e+05,4.007965,88.781248,926.316436,LW


In [68]:
cv_df = final_cv_df.loc[final_cv_df['water_period'].isna()].copy()
metric_df = final_metric_df.loc[final_metric_df['water_period'].isna()].copy()
metric_df

,Model,Group,Feature,Params,r2,mae,mse,mape,exp_var,rmse,water_period
0,KRR,multi_band,nir_red,"{'alpha': 0.001, 'degree': 2, 'gamma': 100.0, ...",0.788071,20.069846,1074.153306,0.413327,0.788071,32.774278,NaN
1,KRR,multi_band,nir_red_green,"{'alpha': 1.5, 'degree': 2, 'gamma': 100.0, 'k...",0.810355,16.427373,961.206772,0.307899,0.819902,31.003335,NaN
2,KRR,multi_band,all_bands,"{'alpha': 1.5, 'degree': 2, 'gamma': 100.0, 'k...",0.795621,15.879604,1035.887372,0.259386,0.812454,32.185204,NaN
3,KRR,multi_band,location,"{'alpha': 1.5, 'degree': 2, 'gamma': 100.0, 'k...",0.731765,20.088438,1359.535351,0.335221,0.753316,36.871878,NaN
4,KRR,multi_band,period,"{'alpha': 0.001, 'degree': 2, 'gamma': 10.0, '...",0.804277,16.946830,992.010443,0.287141,0.804277,31.496197,NaN
...,...,...,...,...,...,...,...,...,...,...,...
304,RF_GEE,multi_band,nir_red_green,"{'maxNodes': None, 'numberOfTrees': 100, 'bagF...",0.863653,14.843933,691.069860,0.324608,0.863864,26.288208,NaN
305,RF_GEE,multi_band,all_bands,"{'maxNodes': None, 'numberOfTrees': 300, 'bagF...",0.888826,12.920339,563.481183,0.275249,0.888867,23.737759,NaN
306,RF_GEE,multi_band,location,"{'maxNodes': None, 'numberOfTrees': 200, 'bagF...",0.942884,8.550017,289.488955,0.182141,0.942884,17.014375,NaN
307,RF_GEE,multi_band,period,"{'maxNodes': None, 'numberOfTrees': 200, 'bagF...",0.945193,8.217552,277.784701,0.149371,0.945201,16.666874,NaN


# Sorting Model Metrics

In [81]:
def sort_models(df, top_n=25):
    """
    Rank models based on best metric performance across multiple criteria.
    
    Scoring logic:
    - Minimize: rmse, mae, mape, mse (lower is better)
    - Maximize: exp_var, r2 (higher is better, r2 capped at 1.0)
    
    Args:
        df: DataFrame with model metrics
        top_n: Return top N models (default 25)
    
    Returns:
        DataFrame with models ranked by best metric count, showing score breakdown
    """
    df = df.copy()
    
    # Drop rows where r2 > 2 (indicates invalid results)
    if 'r2' in df.columns:
        initial_count = len(df)
        df = df[df['r2'] <= 2].copy()
        dropped_count = initial_count - len(df)
        if dropped_count > 0:
            print(f"Dropped {dropped_count} rows with r2 > 2")

    if 'mae' in df.columns:
        initial_count = len(df)
        df = df[df['mae'] <= 40].copy()
        dropped_count = initial_count - len(df)
        if dropped_count > 0:
            print(f"Dropped {dropped_count} rows with mae > 40")
    
    # Define metrics and whether lower is better
    error_metrics = ['rmse', 'mae', 'mape']  # Lower is better
    quality_metrics = ['r2','explained_variance']  # Higher is better
    
    # Get available metrics in dataframe
    available_error = [m for m in error_metrics if m in df.columns]
    available_quality = [m for m in quality_metrics if m in df.columns]
    
    # Initialize score column
    df['best_metric_count'] = 0
    
    # Score based on error metrics (lowest is best)
    for metric in available_error:
        best_value = df[metric].min()
        df.loc[df[metric] == best_value, 'best_metric_count'] += 1
    
    # Score based on quality metrics (highest is best)
    for metric in available_quality:
        best_value = df[metric].max()
        df.loc[df[metric] == best_value, 'best_metric_count'] += 1
    
    # Sort by best metric count (descending), then by MAE (ascending) as tiebreaker
    if 'mae' in df.columns:
        df_sorted = df.sort_values(
            by=['best_metric_count', 'mae'],
            ascending=[False, True]
        ).reset_index(drop=True)
    else:
        df_sorted = df.sort_values(
            by='best_metric_count',
            ascending=False
        ).reset_index(drop=True)
    
    # Return top N models
    return df_sorted.head(top_n)

# Example usage:
best_models = sort_models(cv_df.loc[~cv_df['Feature'].isin(['location','period','period_location'])], top_n=25)
print(best_models[['Model', 'Feature', 'best_metric_count', 'mae', 'r2', 'mape']])

Dropped 5 rows with r2 > 2
Dropped 1 rows with mae > 40
          Model        Feature  best_metric_count        mae        r2  \
0           SVM      all_bands                  1  19.890966  0.668301   
1   polynomial2            nir                  1  21.363015  0.688749   
2           KRR      all_bands                  1  21.730868  0.534705   
3           KRR      all_bands                  1  21.730868  0.534705   
4           KRR      all_bands                  1  21.730868  0.534705   
5        RF_GEE      all_bands                  1  22.266549  0.729764   
6            RF      all_bands                  0  20.121084  0.688698   
7           KRR        nir_red                  0  20.376511  0.697487   
8           KRR        nir_red                  0  20.376511  0.697487   
9           KRR        nir_red                  0  20.376511  0.697487   
10  polynomial2        nir_red                  0  20.426360  0.691890   
11          GBR      all_bands                  0  20.65